# 🏡 Model Selection & Preprocessing Pipelines
## Gurgaon Real Estate Price Prediction

---

# 1. Objective

Following the Feature Selection and Baseline Modeling stages, this notebook evaluates whether advanced preprocessing strategies and ensemble learning algorithms can further improve predictive performance.

All experiments use the finalized feature schema obtained during the Feature Selection phase, ensuring that performance improvements result from preprocessing techniques and model selection rather than changes to the input features.

A primary focus of this notebook is handling the **high-cardinality `sector` feature**, which represents Gurgaon property locations and introduces significant dimensionality when encoded using traditional One-Hot Encoding.

---

# 2. Final Feature Schema

The following features were retained after the Feature Selection phase and are used consistently throughout every preprocessing pipeline.

## Numerical Features

- `bedRoom`
- `bathroom`
- `built_up_area`
- `servant room`
- `store room`

## Categorical Features

- `property_type`
- `sector`
- `balcony`
- `agePossession`
- `furnishing_type`
- `luxury_category`
- `floor_category`

The following features were removed before model training:

- `study room`
- `pooja room`
- `others`

Validation during feature selection confirmed that removing these variables had no measurable impact on predictive performance while simplifying the machine learning pipeline.

---

# 3. Preprocessing Strategies Evaluated

Four preprocessing pipelines were designed and compared.

---

## 3.1 Ordinal Encoding Pipeline

Categorical variables are transformed into sequential integer values using **Ordinal Encoding**.

This approach is computationally efficient and performs well with tree-based ensemble models.

---

## 3.2 Target Encoding Pipeline

Each categorical value is replaced with the **mean target price** of its corresponding category.

For example:

```
Sector 56 → Average Property Price
```

instead of

```
Sector 56 → Category ID
```

Target Encoding enables the model to directly capture pricing trends while avoiding sparse feature spaces.

---

## 3.3 One-Hot Encoding Pipeline

Categorical variables are expanded into binary indicator columns.

Although this encoding avoids introducing artificial ordering, it significantly increases dimensionality, particularly for the `sector` feature.

---

## 3.4 Hybrid One-Hot + PCA Pipeline (Sector-Specific Compression)

To efficiently handle Gurgaon’s high-cardinality sector information, the `sector` feature is isolated into its own preprocessing pipeline.

The workflow is:

```
Sector
      │
      ▼
One-Hot Encoding
      │
      ▼
Principal Component Analysis (95% Variance Retained)
      │
      ▼
Compressed Sector Representation
```

This approach significantly reduces dimensionality while preserving **95% of the original variance**.

The remaining numerical and categorical features are processed using a unified `ColumnTransformer`.

---

# 4. Hybrid Pipeline Architecture

```python
# Mini-pipeline dedicated to the high-cardinality sector feature
pca_sector_transformer = Pipeline([
    ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore')),
    ('pca', PCA(n_components=0.95))
])

# Main preprocessing pipeline
pca_columns_to_encode = [
    'property_type',
    'sector',
    'balcony',
    'agePossession',
    'furnishing_type',
    'luxury_category',
    'floor_category'
]

pca_preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            StandardScaler(),
            ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']
        ),
        (
            'cat',
            OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1),
            pca_columns_to_encode
        ),
        (
            'sector_compressed',
            pca_sector_transformer,
            ['sector']
        ),
        (
            'cat1',
            OneHotEncoder(
                drop='first',
                sparse_output=False,
                handle_unknown='ignore'
            ),
            ['agePossession']
        )
    ],
    remainder='drop'
)
```

---

# 5. Algorithms Evaluated

Three ensemble learning algorithms were selected based on their ability to model complex nonlinear relationships in structured tabular data.

- Random Forest Regressor
- XGBoost Regressor
- ExtraTrees Regressor

---

# 6. Performance Comparison

| Preprocessing Pipeline | Algorithm | R² Score | MAE |
|------------------------|-----------|---------:|----:|
| Ordinal Encoding | Random Forest | **0.88** | **0.49** |
| | XGBoost | **0.89** | **0.47** |
| | ExtraTrees | **0.87** | **0.53** |
| **Target Encoding** ⭐ | Random Forest | **0.90** | **0.43** |
| | XGBoost | **0.90** | **0.47** |
| | ExtraTrees | **0.90** | **0.44** |
| One-Hot Encoding | Random Forest | **0.89** | **0.46** |
| | XGBoost | **0.89** | **0.47** |
| | ExtraTrees | **0.89** | **0.47** |
| One-Hot + PCA | Random Forest | **0.89** | **0.44** |
| | XGBoost | **0.89** | **0.45** |
| | ExtraTrees | **0.89** | **0.45** |

---

# 7. Key Insights

## Baseline Improvement

Every ensemble-based approach matched or outperformed the SVR baseline, demonstrating the effectiveness of combining optimized features with advanced machine learning models.

---

## Impact of Target Encoding

Target Encoding delivered the strongest overall performance.

Advantages include:

- Highest overall **R² (0.90)**
- Lowest prediction error (**MAE = 0.43**)
- Efficient handling of high-cardinality categorical variables
- No sparse feature explosion

---

## Impact of One-Hot + PCA

The hybrid One-Hot + PCA strategy proved highly effective for the `sector` feature.

Benefits include:

- Significant reduction in feature dimensionality
- Preservation of approximately **95%** of the original information
- Stable predictive performance across all ensemble models
- Consistent MAE between **0.44–0.45**

This demonstrates that compressing high-cardinality geographic information improves computational efficiency while maintaining predictive accuracy.

---

# 8. Conclusion

This notebook demonstrates that once an optimized feature set has been established, model performance can be further improved through effective preprocessing and ensemble learning techniques.

Among all evaluated approaches:

- **Target Encoding** achieved the highest predictive accuracy (**R² = 0.90**).
- **Random Forest with Target Encoding** produced the lowest prediction error (**MAE = 0.43**).
- The **One-Hot + PCA** pipeline effectively compressed the high-cardinality `sector` feature while maintaining competitive performance.

Together with the Feature Selection & Engineering notebook and the Baseline Modeling notebook, these experiments establish a complete end-to-end machine learning workflow for accurate Gurgaon real estate price prediction, progressing from feature optimization to baseline validation and finally to high-performing ensemble models.

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_score

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

from sklearn.decomposition import PCA

# Linear Models
from sklearn.linear_model import LinearRegression, Ridge, Lasso

# Support Vector Machines
from sklearn.svm import SVR

# Tree-Based Models
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor, 
    ExtraTreesRegressor, 
    GradientBoostingRegressor, 
    AdaBoostRegressor
)

# Neural Networks (Multi-layer Perceptron)
from sklearn.neural_network import MLPRegressor

# XGBoost (Requires separate installation: pip install xgboost)
from xgboost import XGBRegressor

In [2]:
df = pd.read_csv(r"../data/processed/gurgaon_properties_post_feature_selection_v2.csv")

In [3]:
df.head()

,property_type,sector,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,floor_category,luxury_category,price
0,house,sector 5,2,2,1,Old Property,900.0,0,0,semifurnished,Low-Rise,Standard,0.80
1,flat,sector 104,3,2,3+,Relatively New,1607.0,0,0,unfurnished,High-Rise,Luxury,1.20
2,flat,sector 52,5,7,3+,Moderately Old,5778.0,1,0,semifurnished,Low-Rise,Premium,5.00
3,house,sector 22,7,5,3+,Old Property,2367.0,0,0,unfurnished,Low-Rise,Standard,4.75
4,flat,sohna road,2,2,3,New Property,913.0,0,0,semifurnished,Low-Rise,Premium,0.56


In [4]:
df.shape

(3554, 13)

In [5]:
df.isnull().sum()

property_type      0
sector             0
bedRoom            0
bathroom           0
balcony            0
agePossession      0
built_up_area      0
servant room       0
store room         0
furnishing_type    0
floor_category     0
luxury_category    0
price              0
dtype: int64

In [6]:
df.head()

,property_type,sector,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,floor_category,luxury_category,price
0,house,sector 5,2,2,1,Old Property,900.0,0,0,semifurnished,Low-Rise,Standard,0.80
1,flat,sector 104,3,2,3+,Relatively New,1607.0,0,0,unfurnished,High-Rise,Luxury,1.20
2,flat,sector 52,5,7,3+,Moderately Old,5778.0,1,0,semifurnished,Low-Rise,Premium,5.00
3,house,sector 22,7,5,3+,Old Property,2367.0,0,0,unfurnished,Low-Rise,Standard,4.75
4,flat,sohna road,2,2,3,New Property,913.0,0,0,semifurnished,Low-Rise,Premium,0.56


In [7]:
X = df.drop(columns=['price'])
y = df['price']

In [8]:
# Applying the log1p transformation to the target variable
y_transformed = np.log1p(y)

### Ordinal Encoding 

In [9]:
ordinal_columns_to_encode = ['property_type','sector', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']

In [10]:
# Creating a column transformer for preprocessing
ordinal_preprocessor = ColumnTransformer(
    transformers=[

        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        

        ('cat', OrdinalEncoder(
            handle_unknown='use_encoded_value', 
            unknown_value=-1
        ), ordinal_columns_to_encode)
    ], 

    remainder='passthrough' 
)

In [11]:
# Creating a pipeline
ordinal_pipeline = Pipeline([
    ('preprocessor', ordinal_preprocessor),
    ('regressor', LinearRegression())
])

In [12]:
# K-fold cross-validation
ordinal_kfold = KFold(n_splits=10, shuffle=True, random_state=42)
ordinal_scores = cross_val_score(ordinal_pipeline, X, y_transformed, cv=ordinal_kfold, scoring='r2')

In [13]:
ordinal_scores.mean() , ordinal_scores.std()

(np.float64(0.7175581856226374), np.float64(0.048810112790522504))

In [14]:
ordinal_X_train, ordinal_X_test, ordinal_y_train, ordinal_y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [15]:
ordinal_pipeline.fit(ordinal_X_train , ordinal_y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different tran

In [16]:
ordinal_y_pred = ordinal_pipeline.predict(ordinal_X_test)
ordinal_y_pred = np.expm1(ordinal_y_pred)
mean_absolute_error(np.expm1(ordinal_y_test),ordinal_y_pred)

1.0354277390172524

In [17]:
def ordinal_scorer(ordinal_model_name, ordinal_model):
    
    ordinal_output = []
    
    ordinal_output.append(ordinal_model_name)
    
    ordinal_pipeline = Pipeline([
        ('preprocessor', ordinal_preprocessor),
        ('regressor', ordinal_model)
    ])
    
    # K-fold cross-validation
    ordinal_kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    ordinal_scores = cross_val_score(ordinal_pipeline, X, y_transformed, cv=ordinal_kfold, scoring='r2')
    
    ordinal_output.append(ordinal_scores.mean())
    
    ordinal_X_train, ordinal_X_test, ordinal_y_train, ordinal_y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    ordinal_pipeline.fit(ordinal_X_train,ordinal_y_train)
    
    ordinal_y_pred = ordinal_pipeline.predict(ordinal_X_test)
    
    ordinal_y_pred = np.expm1(ordinal_y_pred)
    
    ordinal_output.append(mean_absolute_error(np.expm1(ordinal_y_test),ordinal_y_pred))
    
    return ordinal_output
    

In [18]:
ordinal_model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting':GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [19]:
ordinal_model_output = []
for ordinal_model_name, ordinal_model in ordinal_model_dict.items():
    ordinal_model_output.append(ordinal_scorer(ordinal_model_name, ordinal_model))

In [20]:
pd.DataFrame(ordinal_model_output , columns=['model_name' , 'r2_score' , 'mae']).sort_values('mae' , ascending=True)

,model_name,r2_score,mae
10,xgboost,0.892325,0.474096
5,random forest,0.882834,0.498268
6,extra trees,0.872250,0.538876
7,gradient boosting,0.874931,0.548740
9,mlp,0.802043,0.679041
4,decision tree,0.762906,0.740017
8,adaboost,0.750401,0.781125
1,svr,0.752250,0.824829
2,ridge,0.717565,1.035205
0,linear_reg,0.717558,1.035428


### One Hot Encoding

In [21]:
onehot_columns_to_encode = ['property_type','sector', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']

onehot_preprocessor = ColumnTransformer(
    transformers=[
       
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        
        
        ('cat', OrdinalEncoder(
            handle_unknown='use_encoded_value', 
            unknown_value=-1
        ), onehot_columns_to_encode),
        
       
        ('cat1', OneHotEncoder(
            drop='first', 
            handle_unknown='ignore',  
            sparse_output=False      
        ), ['sector', 'agePossession', 'furnishing_type'])
    ], 
   
    remainder='passthrough'
)

In [22]:
# Creating a pipeline
onehot_pipeline = Pipeline([
    ('preprocessor', onehot_preprocessor),
    ('regressor', LinearRegression())
])

In [23]:
# K-fold cross-validation
onehot_kfold = KFold(n_splits=10, shuffle=True, random_state=42)
onehot_scores = cross_val_score(onehot_pipeline, X, y_transformed, cv=onehot_kfold, scoring='r2')

C:\Users\Predator\AppData\Roaming\Python\Python313\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\Predator\AppData\Roaming\Python\Python313\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [24]:
onehot_scores.mean() , onehot_scores.std()

(np.float64(0.843510576173507), np.float64(0.02107174546502739))

In [25]:
onehot_X_train, onehot_X_test, onehot_y_train, onehot_y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [26]:
onehot_pipeline.fit(onehot_X_train , onehot_y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different

In [27]:
onehot_y_pred = onehot_pipeline.predict(onehot_X_test)
onehot_y_pred = np.expm1(onehot_y_pred)
mean_absolute_error(np.expm1(onehot_y_test),onehot_y_pred)

0.667200176715685

In [28]:
def onehot_scorer(onehot_model_name, onehot_model):
    
    onehot_output = []
    
    onehot_output.append(onehot_model_name)
    
    onehot_pipeline = Pipeline([
        ('preprocessor', onehot_preprocessor),
        ('regressor', onehot_model)
    ])
    
    # K-fold cross-validation
    onehot_kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    onehot_scores = cross_val_score(onehot_pipeline, X, y_transformed, cv=onehot_kfold, scoring='r2')
    
    onehot_output.append(onehot_scores.mean())
    
    onehot_X_train, onehot_X_test, onehot_y_train, onehot_y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    onehot_pipeline.fit(onehot_X_train,onehot_y_train)
    
    onehot_y_pred = onehot_pipeline.predict(onehot_X_test)
    
    onehot_y_pred = np.expm1(onehot_y_pred)
    
    onehot_output.append(mean_absolute_error(np.expm1(onehot_y_test),onehot_y_pred))
    
    return onehot_output

In [29]:
onehot_model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting':GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [30]:
onehot_model_output = []
for onehot_model_name, onehot_model in onehot_model_dict.items():
    onehot_model_output.append(onehot_scorer(onehot_model_name, onehot_model))

C:\Users\Predator\AppData\Roaming\Python\Python313\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\Predator\AppData\Roaming\Python\Python313\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\Predator\AppData\Roaming\Python\Python313\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\Predator\AppData\Roaming\Python\Python313\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be

In [31]:
pd.DataFrame(onehot_model_output , columns=['model_name' , 'r2_score' , 'mae']).sort_values('mae' , ascending=True)

,model_name,r2_score,mae
5,random forest,0.891854,0.465640
10,xgboost,0.894456,0.472739
6,extra trees,0.893277,0.473049
7,gradient boosting,0.876330,0.527445
9,mlp,0.866927,0.557868
0,linear_reg,0.843511,0.667200
2,ridge,0.844099,0.675554
4,decision tree,0.793842,0.676252
8,adaboost,0.757863,0.769970
1,svr,0.756734,0.818955


### Target Encoder

In [32]:
import category_encoders as ce

target_columns_to_encode = ['property_type','sector', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']

# Creating a column transformer for preprocessing
target_preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), target_columns_to_encode),
        ('cat1',OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'),['agePossession']),
        ('target_enc', ce.TargetEncoder(), ['sector'])
    ], 
    remainder='passthrough'
)

In [33]:
# Creating a pipeline
target_pipeline = Pipeline([
    ('preprocessor', target_preprocessor),
    ('regressor', LinearRegression())
])

In [34]:
# K-fold cross-validation
target_kfold = KFold(n_splits=10, shuffle=True, random_state=42)
target_scores = cross_val_score(target_pipeline, X, y_transformed, cv=target_kfold, scoring='r2')

In [35]:
target_scores.mean(),target_scores.std()

(np.float64(0.814297573154529), np.float64(0.031123109394899495))

In [36]:
def target_scorer(target_model_name, target_model):
    
    target_output = []
    
    target_output.append(target_model_name)
    
    target_pipeline = Pipeline([
        ('preprocessor', target_preprocessor),
        ('regressor', target_model)
    ])
    
    # K-fold cross-validation
    target_kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    target_scores = cross_val_score(target_pipeline, X, y_transformed, cv=target_kfold, scoring='r2')
    
    target_output.append(target_scores.mean())
    
    target_X_train, target_X_test, target_y_train, target_y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    target_pipeline.fit(target_X_train,target_y_train)
    
    target_y_pred = target_pipeline.predict(target_X_test)
    
    target_y_pred = np.expm1(target_y_pred)
    
    target_output.append(mean_absolute_error(np.expm1(target_y_test),target_y_pred))
    
    return target_output
    

In [37]:
target_model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting':GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [38]:
target_model_output = []
for target_model_name, target_model in target_model_dict.items():
    target_model_output.append(target_scorer(target_model_name, target_model))

In [39]:
pd.DataFrame(target_model_output , columns=['model_name' , 'r2_score' , 'mae']).sort_values('mae' , ascending=True)

,model_name,r2_score,mae
5,random forest,0.903234,0.435123
6,extra trees,0.901525,0.446647
10,xgboost,0.903113,0.470663
7,gradient boosting,0.888791,0.498740
9,mlp,0.842562,0.622409
4,decision tree,0.820473,0.623312
8,adaboost,0.818411,0.686315
0,linear_reg,0.814298,0.748818
2,ridge,0.814316,0.749446
1,svr,0.769389,0.800492


### One Hot Encoding with PCA

In [40]:
# 1. Create a mini-pipeline dedicated ONLY to encoding and compressing the sector
pca_sector_transformer = Pipeline([
    ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore')),
    ('pca', PCA(n_components=0.95)) # Compresses only the sector columns
])

# 2. Feed it into your main preprocessor
pca_columns_to_encode = ['property_type','sector', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']
pca_preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), pca_columns_to_encode),
        
        # Target only the sector for dimensionality reduction
        ('sector_compressed', pca_sector_transformer, ['sector']), 
        
        ('cat1', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), ['agePossession'])
    ], 
    remainder='drop'
)

# Your main pipeline stays clean and linear
pca_pipeline = Pipeline([
    ('preprocessor', pca_preprocessor),
    ('regressor', LinearRegression())
])

In [41]:
# K-fold cross-validation
pca_kfold = KFold(n_splits=10, shuffle=True, random_state=42)
pca_scores = cross_val_score(pca_pipeline, X, y_transformed, cv=pca_kfold, scoring='r2')

In [42]:
pca_scores.mean(),pca_scores.std()

(np.float64(0.8322166266536497), np.float64(0.025885706584956276))

In [43]:
def pca_scorer(pca_model_name, pca_model):
    
    pca_output = []
    
    pca_output.append(pca_model_name)
    
    pca_pipeline = Pipeline([
        ('preprocessor', pca_preprocessor),
        ('regressor', pca_model)
    ])
    
    # K-fold cross-validation
    pca_kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    pca_scores = cross_val_score(pca_pipeline, X, y_transformed, cv=pca_kfold, scoring='r2')
    
    pca_output.append(pca_scores.mean())
    
    pca_X_train, pca_X_test, pca_y_train, pca_y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pca_pipeline.fit(pca_X_train,pca_y_train)
    
    pca_y_pred = pca_pipeline.predict(pca_X_test)
    
    pca_y_pred = np.expm1(pca_y_pred)
    
    pca_output.append(mean_absolute_error(np.expm1(pca_y_test),pca_y_pred))
    
    return pca_output
    

In [44]:
pca_model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting':GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [45]:
pca_model_output = []
for pca_model_name, pca_model in pca_model_dict.items():
    pca_model_output.append(pca_scorer(pca_model_name, pca_model))

In [46]:
pd.DataFrame(pca_model_output , columns=['model_name' , 'r2_score' , 'mae']).sort_values('mae' , ascending=True)

,model_name,r2_score,mae
5,random forest,0.896422,0.446363
10,xgboost,0.898674,0.455251
6,extra trees,0.895184,0.459714
7,gradient boosting,0.881255,0.512487
9,mlp,0.865725,0.527576
4,decision tree,0.813974,0.544362
0,linear_reg,0.832217,0.683074
2,ridge,0.832459,0.691715
8,adaboost,0.763060,0.786761
1,svr,0.756108,0.820966
